In [7]:
import pandas as pd
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster


In [2]:
view1 = pd.read_csv('clusters/GO_BP-clusters.csv')
view2 = pd.read_csv('clusters/GO_MF-clusters.csv')
view3 = pd.read_csv('clusters/GO_CC-clusters.csv')
view4 = pd.read_csv('clusters/HPO-clusters.csv')
mofa = pd.read_csv('clusters/mofa-clusters.csv')

clusterings = {
    'GO_BP': view1,
    'GO_MF': view2,
    'GO_CC': view3,
    'HPO': view4,
    'MOFA': mofa
}

In [8]:
def build_coassignment_matrix(clusterings, views_to_include=None, ks_to_include=None):
    """
    Build the co-assignment matrix from multiple clusterings.

    Parameters
    ----------
    clusterings : dict
        {view_name: DataFrame with shape (n_samples, n_k)}
    views_to_include : list of str, optional
        Which views to include (default all)
    ks_to_include : list of int, optional
        Which k values to include (default all ks in clusterings)

    Returns
    -------
    coassign : np.ndarray, shape (n_samples, n_samples)
        Fraction of clusterings in which each pair of samples is in the same cluster
    """
    if views_to_include is None:
        views_to_include = list(clusterings.keys())

    # Determine ks
    if ks_to_include is None:
        # use all columns in first view
        ks_to_include = [int(c) for c in clusterings[views_to_include[0]].columns]

    n_samples = clusterings[views_to_include[0]].shape[0]
    coassign = np.zeros((n_samples, n_samples), dtype=float)
    n_total = 0

    for view in views_to_include:
        df = clusterings[view]
        for k in ks_to_include:
            labels = df[str(k)].values.ravel()
            # build co-assignment for this clustering
            for i in range(n_samples):
                for j in range(i, n_samples):
                    if labels[i] == labels[j]:
                        coassign[i, j] += 1
                        if i != j:
                            coassign[j, i] += 1
            n_total += 1

    coassign /= n_total
    return coassign


def consensus_clustering(coassign_matrix, n_clusters=10, method='average'):
    """
    Compute consensus clusters using hierarchical clustering.

    Parameters
    ----------
    coassign_matrix : np.ndarray
        n_samples x n_samples co-assignment matrix
    n_clusters : int
        Number of consensus clusters
    method : str
        Linkage method for hierarchical clustering (default 'average')

    Returns
    -------
    labels : np.ndarray, shape (n_samples,)
        Consensus cluster labels (1..n_clusters)
    """
    # Convert similarity to distance
    distance = 1.0 - coassign_matrix

    # Condensed distance matrix for linkage
    from scipy.spatial.distance import squareform
    dist_condensed = squareform(distance, checks=False)

    Z = linkage(dist_condensed, method=method)
    labels = fcluster(Z, t=n_clusters, criterion='maxclust')
    return labels

In [9]:
# Step 1: build co-assignment matrix
coassign = build_coassignment_matrix(clusterings,
                                     views_to_include=["HPO","GO_CC","GO_BP","GO_MF","MOFA"],
                                     ks_to_include=range(20,51))  # pick stable ks

# Step 2: compute consensus clusters
n_consensus_clusters = 10
consensus_labels = consensus_clustering(coassign, n_clusters=n_consensus_clusters)

# Step 3: view results
print("Consensus cluster counts:")
print(pd.Series(consensus_labels).value_counts())

Consensus cluster counts:
10    1840
9     1343
1      554
5      334
2      314
7      239
8      175
6      162
4      124
3       98
Name: count, dtype: int64
